In [1]:


"""
End-to-end runner: train MAPPO, evaluate MAPPO-only vs hybrid MAPPO+MPC,
run parameter sweep, and produce all thesis-quality presentation figures.

Comparison architecture:
  - MAPPO only: stochastic policy with actuator noise (no constraint guarantees)
  - MAPPO + MPC: MPC as low-level controller with constraint guarantees

Run:    python main.py
"""
import os
import sys
import argparse
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
from environment import GardenEnvironment
from mappo import MAPPO
from mpc_controller import MPCController
import train as train_mod
import evaluate as eval_mod
import visualize as viz


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--out_dir", type=str, default="results")
    p.add_argument("--n_updates", type=int, default=200)
    p.add_argument("--episodes_per_update", type=int, default=4)
    p.add_argument("--n_eval_eps", type=int, default=3)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--skip_train", action="store_true",
                   help="Skip training, just evaluate from out_dir/checkpoint")
    args,_ = p.parse_known_args()

    out = args.out_dir
    os.makedirs(out, exist_ok=True)

    # ------------------------------------------------------------------
    # 1) Training
    # ------------------------------------------------------------------
    if not args.skip_train:
        print("=" * 70)
        print("PHASE 1: Training MAPPO with boustrophedon waypoints")
        print("=" * 70)
        train_args, _ = train_mod.get_parser().parse_known_args([
            "--out_dir", out,
            "--seed", str(args.seed),
            "--n_updates", str(args.n_updates),
            "--episodes_per_update", str(args.episodes_per_update),
            "--log_every", "5",
        ])
        train_mod.train(train_args)

    # ------------------------------------------------------------------
    # 2) Evaluation: MAPPO-only vs MAPPO+MPC
    # ------------------------------------------------------------------
    print()
    print("=" * 70)
    print("PHASE 2: Evaluation — MAPPO only vs MAPPO + MPC")
    print("=" * 70)
    eval_args, _ = eval_mod.get_parser().parse_known_args([
        "--in_dir", out,
        "--seed", "1000",
        "--n_eval_eps", str(args.n_eval_eps),
    ])
    results, env = eval_mod.evaluate(eval_args)

    # ------------------------------------------------------------------
    # 3) Presentation figures
    # ------------------------------------------------------------------
    print()
    print("=" * 70)
    print("PHASE 3: Producing thesis presentation figures")
    print("=" * 70)
    hist_path = os.path.join(out, "history.npz")
    viz.plot_reward_curve(hist_path, os.path.join(out, "reward_curve.png"))
    viz.plot_training_summary(hist_path, os.path.join(out, "training_summary.png"))
    viz.plot_paths_comparison(results, env, os.path.join(out, "paths_comparison.png"), episode_idx=0)
    viz.plot_coverage_heatmap(results, env, os.path.join(out, "coverage_heatmap.png"), episode_idx=0)
    viz.plot_coverage_detail(results, env, os.path.join(out, "coverage_detail.png"), episode_idx=0)
    viz.plot_comparison_metrics(results, os.path.join(out, "comparison_metrics.png"))
    print(f"  Saved reward_curve.png, training_summary.png, paths_comparison.png, "
          f"coverage_heatmap.png, coverage_detail.png, comparison_metrics.png")

    # ------------------------------------------------------------------
    # 4) Parameter sweep
    # ------------------------------------------------------------------
    print()
    print("=" * 70)
    print("PHASE 4: Parameter sweep")
    print("=" * 70)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    sweep_settings = {
        "8 trees, N=15":    dict(n_trees=4, mpc_horizon=15),
        "10 trees, N=15":       dict(n_trees=5, mpc_horizon=15),
        "12 trees, N=15":      dict(n_trees=6, mpc_horizon=15),
        " 8 trees N=20":      dict(n_trees=4, mpc_horizon=20),
    }
    sweep_results = {}
    for name, cfg in sweep_settings.items():
        env_s = GardenEnvironment(
            n_trees_per_region=cfg["n_trees"],
            spray_radius=2.5,
            lane_spacing=4.0,
            max_steps=1500,
            max_accel=3.5,
            seed=args.seed,
        )
        agent_s = MAPPO(obs_dim=env_s.obs_dim, act_dim=env_s.act_dim,
                        n_agents=env_s.n_agents, device=device)
        agent_s.load(os.path.join(out, "mappo.pt"))
        mpc_s = MPCController(
            horizon=cfg["mpc_horizon"], dt=env_s.dt, u_max=env_s.max_accel,
            tree_radius=env_s.tree_radius, tree_safety_margin=0.5,
            inter_drone_min=env_s.inter_drone_min,
            boundary=(0.0, env_s.size),
        )
        per = {"with_mpc": [], "without_mpc": []}
        for ep in range(args.n_eval_eps):
            env_s.rng = np.random.default_rng(2000 + ep)
            per["with_mpc"].append(
                eval_mod.run_eval_episode(env_s, agent_s, use_mpc=True, mpc=mpc_s,
                                          record_path=False, act_noise_scale=0.0)
            )
            env_s.rng = np.random.default_rng(2000 + ep)
            per["without_mpc"].append(
                eval_mod.run_eval_episode(env_s, agent_s, use_mpc=False, mpc=None,
                                          record_path=False, act_noise_scale=0.0)
            )
        sweep_results[name] = per
        print(
            f"  {name:20s}  "
            f"MAPPO+MPC: cov={np.mean([r['coverage'] for r in per['with_mpc']]):.3f}  "
            f"MAPPO only: cov={np.mean([r['coverage'] for r in per['without_mpc']]):.3f}"
        )

    viz.plot_parameter_sweep(sweep_results, os.path.join(out, "parameter_sweep.png"))
    print(f"  Saved parameter_sweep.png")

    print()
    print("Done. All figures saved in", out)


if __name__ == "__main__":
    main()


PHASE 1: Training MAPPO with boustrophedon waypoints
upd    1/200  ep     4  R=6677.53  len=1500.0  cov=0.165  pi_loss=-0.002  v_loss=24719.63  t=4s
upd    5/200  ep    20  R=23929.99  len=1500.0  cov=0.710  pi_loss=-0.001  v_loss=149823.45  t=19s
upd   10/200  ep    40  R=29725.09  len=1500.0  cov=0.854  pi_loss=-0.000  v_loss=244486.81  t=38s
upd   15/200  ep    60  R=33571.92  len=1500.0  cov=0.891  pi_loss=0.000  v_loss=293441.66  t=56s
upd   20/200  ep    80  R=35330.19  len=1500.0  cov=0.879  pi_loss=-0.000  v_loss=382138.21  t=75s
upd   25/200  ep   100  R=34972.67  len=1500.0  cov=0.940  pi_loss=0.000  v_loss=371288.12  t=95s
upd   30/200  ep   120  R=30992.56  len=1500.0  cov=0.912  pi_loss=-0.001  v_loss=274387.02  t=114s
upd   35/200  ep   140  R=31707.50  len=1500.0  cov=0.955  pi_loss=-0.001  v_loss=344188.46  t=132s
upd   40/200  ep   160  R=32143.75  len=1500.0  cov=0.950  pi_loss=0.000  v_loss=261632.29  t=151s
upd   45/200  ep   180  R=33275.19  len=1500.0  cov=0.968  

/home/student15/mpc_controller.py:195: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  self._prob.solve(solver=cp.OSQP, warm_start=True, verbose=False,


ep 2/3  MAPPO+MPC: R=65518.8 cov=0.985 col= 1 len=1500 mpc_used=1500 || MAPPO only: R=33193.3 cov=0.924 col=29 len=1500
ep 3/3  MAPPO+MPC: R=65304.9 cov=0.968 col= 0 len=1500 mpc_used=1500 || MAPPO only: R=29604.2 cov=0.874 col=43 len=1500

Summary over 3 episodes:
  MAPPO+MPC:  coverage=0.974 +/- 0.008  collisions=0.3
  MAPPO only: coverage=0.911 +/- 0.027  collisions=34.3

PHASE 3: Producing thesis presentation figures
  Saved reward_curve.png, training_summary.png, paths_comparison.png, coverage_heatmap.png, coverage_detail.png, comparison_metrics.png

PHASE 4: Parameter sweep


/home/student15/mpc_controller.py:195: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  self._prob.solve(solver=cp.OSQP, warm_start=True, verbose=False,


  8 trees, N=15         MAPPO+MPC: cov=0.971  MAPPO only: cov=0.904
  10 trees, N=15        MAPPO+MPC: cov=0.967  MAPPO only: cov=0.891
  12 trees, N=15        MAPPO+MPC: cov=0.962  MAPPO only: cov=0.919
   8 trees N=20         MAPPO+MPC: cov=0.970  MAPPO only: cov=0.881
  Saved parameter_sweep.png

Done. All figures saved in results
